# Fase 7 — Modelo de datos, SQLite y preparación para Power BI

Esta fase transforma el histórico y el forecast final de la Fase 6 en una capa analítica reproducible.

**Objetivos:**

- construir un modelo en estrella con dimensiones de fecha y tienda;
- separar ventas reales y forecast en dos tablas de hechos;
- crear una base SQLite y vistas SQL reutilizables;
- exportar tablas limpias para Power BI;
- reconciliar todos los resultados con la Fase 6 antes del commit.

In [1]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd
from IPython.display import display

CURRENT = Path.cwd().resolve()
REPO_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

Repository root: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting


## 1. Construcción de la capa analítica

La lógica reutilizable está en `src/build_sql_model.py`. El notebook la ejecuta sin duplicar transformaciones.

In [2]:
from src.build_sql_model import build_analytics_model

result = build_analytics_model(REPO_ROOT)

print(f"SQLite database: {result['database_path']}")
print(f"Power BI folder: {REPO_ROOT / 'powerbi' / 'data'}")

SQLite database: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting\data\processed\rossmann_analytics.db
Power BI folder: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting\powerbi\data


## 2. Revisión de controles

Todos los controles deben aparecer con `Passed = 1`. No se debe hacer commit si alguno falla.

In [3]:
quality_results = result["quality_results"].copy()
display(quality_results)

passed = int(quality_results["Passed"].sum())
total = len(quality_results)
print(f"Controls passed: {passed}/{total}")

assert passed == total, "There are failed Phase 7 controls."

,Area,ControlName,Passed,ObservedValue,ExpectedValue,Tolerance,CheckedAtUTC
0,Population,Historical rows loaded,1,1017209,1017209,Exact,2026-06-28T00:22:09.864582+00:00
1,Population,Forecast rows loaded,1,41088,41088,Exact,2026-06-28T00:22:09.864582+00:00
2,Dimensions,Store dimension rows,1,1115,1115,Exact,2026-06-28T00:22:09.864582+00:00
3,Dimensions,Date dimension is continuous,1,990,990,Exact,2026-06-28T00:22:09.864582+00:00
4,Keys,Historical Store-Date duplicates,1,0,0,Exact,2026-06-28T00:22:09.864582+00:00
5,Keys,Forecast Store-Date duplicates,1,0,0,Exact,2026-06-28T00:22:09.864582+00:00
6,Integrity,Historical stores found in dimension,1,0,0,Exact,2026-06-28T00:22:09.864582+00:00
7,Integrity,Forecast stores found in dimension,1,0,0,Exact,2026-06-28T00:22:09.864582+00:00
8,Integrity,Historical dates found in dimension,1,0,0,Exact,2026-06-28T00:22:09.864582+00:00
9,Integrity,Forecast dates found in dimension,1,0,0,Exact,2026-06-28T00:22:09.864582+00:00


Controls passed: 24/24


## 3. Tamaño y cobertura de las tablas

In [4]:
database_path = result["database_path"]

coverage_query = """
SELECT 'dim_date' AS table_name, COUNT(*) AS rows, NULL AS stores, MIN(Date) AS start_date, MAX(Date) AS end_date
FROM dim_date
UNION ALL
SELECT 'dim_store', COUNT(*), COUNT(*), NULL, NULL
FROM dim_store
UNION ALL
SELECT 'fact_sales_history', COUNT(*), COUNT(DISTINCT Store), MIN(d.Date), MAX(d.Date)
FROM fact_sales_history h JOIN dim_date d ON d.DateKey = h.DateKey
UNION ALL
SELECT 'fact_sales_forecast', COUNT(*), COUNT(DISTINCT Store), MIN(d.Date), MAX(d.Date)
FROM fact_sales_forecast f JOIN dim_date d ON d.DateKey = f.DateKey;
"""

with sqlite3.connect(database_path) as connection:
    coverage = pd.read_sql_query(coverage_query, connection)

display(coverage)

,table_name,rows,stores,start_date,end_date
0,dim_date,990,NaN,2013-01-01,2015-09-17
1,dim_store,1115,1115.0,None,None
2,fact_sales_history,1017209,1115.0,2013-01-01,2015-07-31
3,fact_sales_forecast,41088,856.0,2015-08-01,2015-09-17


## 4. Métricas del modelo registradas

In [5]:
with sqlite3.connect(database_path) as connection:
    model_metrics = pd.read_sql_query("SELECT * FROM model_metrics", connection)

display(model_metrics.T.rename(columns={0: "value"}))

,value
ModelKey,1
Project,Rossmann Retail Sales Forecasting & FP&A Analy...
ModelName,Recent 365-day Store + weekday + Promo
ModelType,Rule-based historical average
PredictionColumn,BaselineRecent365
HistoryWindowDays,365
HistoricalDataStart,2013-01-01
HistoricalDataEnd,2015-07-31
ModelLookbackStart,2014-08-01
ForecastStart,2015-08-01


## 5. Consultas SQL de validación y análisis

In [6]:
with sqlite3.connect(database_path) as connection:
    daily_forecast = pd.read_sql_query(
        "SELECT * FROM vw_forecast_daily_summary ORDER BY Date LIMIT 10",
        connection,
    )
    top_stores = pd.read_sql_query(
        "SELECT * FROM vw_forecast_store_summary ORDER BY ForecastSales DESC LIMIT 10",
        connection,
    )
    store_types = pd.read_sql_query(
        "SELECT * FROM vw_forecast_by_store_type ORDER BY ForecastSales DESC",
        connection,
    )

print("First forecast dates")
display(daily_forecast)
print("Top forecast stores")
display(top_stores)
print("Forecast by store type and assortment")
display(store_types)

First forecast dates


,DateKey,Date,ReportingStores,OpenStores,PromoStoreDays,ForecastSales,MeanSalesPerOpenStore
0,20150801,2015-08-01,856,856,0,5.351691e+06,6251.975809
1,20150802,2015-08-02,856,27,0,2.065069e+05,7648.404358
2,20150803,2015-08-03,856,856,856,8.196577e+06,9575.440164
3,20150804,2015-08-04,856,856,856,6.999309e+06,8176.763246
4,20150805,2015-08-05,856,856,856,6.525117e+06,7622.800148
5,20150806,2015-08-06,856,856,856,6.509286e+06,7604.306388
6,20150807,2015-08-07,856,856,856,6.670141e+06,7792.220942
7,20150808,2015-08-08,856,848,0,5.305819e+06,6256.861620
8,20150809,2015-08-09,856,27,0,2.065069e+05,7648.404358
9,20150810,2015-08-10,856,856,0,5.338624e+06,6236.710096


Top forecast stores


,Store,StoreType,Assortment,Promo2,ForecastSales,OpenDays,PromoDays,MeanCalendarDaySales,MaximumDailySales,ClosedDays,MeanSalesPerOpenDay
0,262,b,a,0,997985.449204,48,19,20791.363525,29063.134615,0,20791.363525
1,1114,a,c,0,877611.835289,41,19,18283.579902,24195.758621,7,21405.166714
2,562,b,c,0,866713.515344,48,19,18056.531570,21657.862069,0,18056.531570
3,251,a,c,0,799913.903940,41,19,16664.872999,24692.517241,7,19510.095218
4,842,d,c,0,794340.710219,40,19,16548.764796,23795.470588,8,19858.517755
5,733,b,b,0,736434.801211,48,19,15342.391692,16787.433333,0,15342.391692
6,383,a,c,0,707005.072373,41,19,14729.272341,20632.344828,7,17244.026155
7,756,a,c,1,706095.054096,41,19,14710.313627,21076.586207,7,17221.830588
8,335,b,a,1,665326.051586,48,19,13860.959408,22040.689655,0,13860.959408
9,586,a,c,0,653161.136689,41,19,13607.523681,19993.392857,7,15930.759431


Forecast by store type and assortment


,StoreType,Assortment,Stores,OpenStoreDays,PromoStoreDays,ForecastSales,MeanSalesPerOpenStoreDay
0,a,a,264,10756,5012,6.979606e+07,6489.035198
1,a,c,197,8147,3743,6.247560e+07,7668.541152
2,d,c,182,7440,3458,5.438986e+07,7310.465375
3,d,a,112,4559,2124,2.960694e+07,6494.174585
4,c,a,45,1836,855,1.235644e+07,6730.087963
5,c,c,44,1799,836,1.190206e+07,6615.928688
6,b,b,9,423,167,3.977124e+06,9402.183251
7,b,a,2,96,38,1.663312e+06,17326.161467
8,b,c,1,48,19,8.667135e+05,18056.531570


## 6. Archivos preparados para Power BI

In [7]:
exports = []
for name, path in result["powerbi_exports"].items():
    exports.append({
        "table": name,
        "path": str(path.relative_to(REPO_ROOT)),
        "size_mb": round(path.stat().st_size / 1_048_576, 2),
    })

display(pd.DataFrame(exports))

,table,path,size_mb
0,dim_date,powerbi\data\dim_date.csv,0.06
1,dim_store,powerbi\data\dim_store.csv,0.06
2,fact_sales_history,powerbi\data\fact_sales_history.csv,31.44
3,fact_sales_forecast,powerbi\data\fact_sales_forecast.csv,4.23
4,model_metrics,powerbi\data\model_metrics.csv,0.00
5,data_quality_results,powerbi\data\data_quality_results.csv,0.00


## Conclusión provisional

La Fase 7 queda técnicamente preparada cuando:

- todos los controles están aprobados;
- la base `data/processed/rossmann_analytics.db` se ha creado;
- las dimensiones y hechos tienen la granularidad esperada;
- las reconciliaciones con los resúmenes de la Fase 6 son exactas;
- los seis CSV de `powerbi/data/` están disponibles.

**No hacer commit todavía.** Primero se revisarán conjuntamente los resultados de este notebook.